# Regimenes climaticos y pronostico multitarea

Pronostico semanal de dengue para 2025. Cada ano se entrena solo con lo anterior. Los departamentos no entran por una lista fija: un modelo de mezclas gaussianas les asigna probabilidades a partir del clima, la altitud implicita en la presion, la fase estacional y la escala. Esas probabilidades pesan el apilado final.

El entrenamiento vive en `modelo.py` (`ejecutar`). Este cuaderno relee ese resultado si ya esta al dia y muestra la validacion.


## Agrupamiento

Para el corte de 2025 el modelo ve solo 2007-2024. El numero de grupos sale del BIC entre 2 y 4 componentes. Lo que se usa despues no es la etiqueta dura, sino P(grupo | departamento).


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import modelo

train, test, _ = modelo._horizonte().cargar()
grupos = modelo.agrupar(train[train.anio < 2025])
nombres = train.drop_duplicates("id_departamento").set_index("id_departamento").departamento
tabla = grupos.copy()
tabla.insert(0, "departamento", tabla.index.map(nombres))
tabla = tabla.sort_values(["p0", "p1", "p2"], ascending=False)
tabla.round(3)


,departamento,p0,p1,p2,p3
id_departamento,,,,,
2,Antioquia,1.0,0.000,0.000,0.0
27,Santander,1.0,0.000,0.000,0.0
21,Narino,1.0,0.000,0.000,0.0
22,Norte De Santander,1.0,0.000,0.000,0.0
30,Valle Del Cauca,1.0,0.000,0.000,0.0
29,Tolima,1.0,0.000,0.000,0.0
10,Cauca,1.0,0.000,0.000,0.0
18,Huila,1.0,0.000,0.000,0.0
7,Caldas,1.0,0.000,0.000,0.0


## Familias y compuerta

- Familia A: mezcla convexa, por bloque de horizonte, de las bases historicas. Si un brote largo ya viene cayendo, las bases recientes se acercan a la mediana de largo plazo. La fuerza de ese acercamiento se elige con los anos anteriores.
- Familia B: LightGBM, XGBoost y HistGradientBoosting predicen el log-ratio contra la familia A. No usan el id del departamento.
- Familia C: TCN, MLP y GRU a 52 semanas, con la misma compuerta de la red anterior. La ancla de persistencia usa la carga de 104 semanas y la razon cierre/pico, asi un diciembre alto no se arrastra cuando el brote ya se agoto.
- Meta: una mezcla convexa distinta en cada grupo, ponderada por P(grupo). Despues un empalme H1-H2 y un encogimiento de Bayes del efecto del primer miercoles del mes.

La referencia a batir es la B de horizontes: H1 3534 / 3909 y H2 5164 / 5934 (2014-2023 y 2016-2023).


In [2]:
resultados = Path("resultados.json")
codigo = Path(modelo.__file__).stat().st_mtime
if not resultados.exists() or resultados.stat().st_mtime < codigo:
    metricas = modelo.ejecutar()
else:
    metricas = json.loads(resultados.read_text(encoding="utf-8"))

filas = []
for nombre, valores in {"referencia": metricas["referencia_b"], **metricas["familias"], "meta": metricas["meta"]}.items():
    filas.append({"modelo": nombre, **{k: round(v, 1) for k, v in valores.items()}})
pd.DataFrame(filas).set_index("modelo")


,h1_2014,h1_2016,h2_2014,h2_2016
modelo,,,,
referencia,3534.1,3909.4,5163.8,5933.7
a,4272.0,4709.7,5349.2,6173.4
b,4176.0,4609.3,5472.5,6221.2
mlp,4863.6,5580.7,6101.5,7199.4
tcn,4634.6,5094.5,6296.4,7168.0
gru,4451.1,4999.9,5345.7,6188.5
meta,4180.7,4595.5,5348.1,6172.0


## Ano por ano

MSE de la meta contra la B de horizontes. Un ano negativo en la diferencia significa que la meta gano ese ano.


In [3]:
pred = pd.read_parquet("predicciones.parquet")
ref = pd.read_parquet(modelo.RAIZ / "ensayos/juan/2026-11-04-red-dinamica-h1/b_walkforward.parquet")
comparacion = []
for anio in range(2014, 2025):
    real = pred[pred.anio == anio]
    base = ref.loc[ref.anio == anio, ["id_departamento", "semana", "b"]].merge(
        real[["id_departamento", "semana", "y", "meta"]], on=["id_departamento", "semana"]
    )
    for mitad, mascara in (("h1", base.semana <= 26), ("h2", base.semana >= 27)):
        y = base.loc[mascara, "y"].to_numpy(float)
        comparacion.append({
            "anio": anio,
            "mitad": mitad,
            "meta": round(float(np.mean((y - base.loc[mascara, "meta"]) ** 2)), 1),
            "b": round(float(np.mean((y - base.loc[mascara, "b"]) ** 2)), 1),
        })
anual = pd.DataFrame(comparacion)
anual["meta_menos_b"] = (anual.meta - anual.b).round(1)
anual


,anio,mitad,meta,b,meta_menos_b
0,2014,h1,1799.1,1418.9,380.2
1,2014,h2,2178.1,2067.6,110.5
2,2015,h1,3243.8,2647.2,596.6
3,2015,h2,1926.6,2100.1,-173.5
4,2016,h1,10237.7,9742.7,495.0
5,2016,h2,11064.8,8117.5,2947.3
6,2017,h1,2567.2,1120.2,1447.0
7,2017,h2,2759.4,967.5,1791.9
8,2018,h1,400.4,415.6,-15.2
9,2018,h2,1376.4,1504.6,-128.2


## Diagnostico de 2025, sin usar el caso real

Las medias de la Costa, Valle y el resto salen de las sondas ya calificadas, no de una lista escrita en el modelo. El objetivo de la Costa es cerca de 0,95 veces B, el de Valle cerca de la mitad de B y el del resto un poco por encima de B. Sirve para ver si el pronostico repite el arrastre de diciembre.


In [4]:
diag = metricas["diagnostico_2025"]
lectura = pd.DataFrame({
    "b": diag["media_b"],
    "meta": diag["media_modelo"],
})
lectura["meta_sobre_b"] = (lectura.meta / lectura.b).round(3)
lectura.round(1)


,b,meta,meta_sobre_b
costa,105.7,119.7,1.1
valle,192.9,211.7,1.1
resto,37.7,42.9,1.1


In [5]:
envio = pd.read_csv("submission.csv")
muestra = pd.read_csv(modelo.RAIZ / "data/sample_submission.csv")
pd.Series({
    "filas": len(envio),
    "columnas": ",".join(envio.columns),
    "mismo_orden": bool(envio["id"].tolist() == muestra["id"].tolist()),
    "minimo": float(envio.dengue.min()),
    "media_h1_2025": round(metricas["media_h1_2025"], 2),
    "media_h2_2025": round(metricas["media_h2_2025"], 2),
})


filas                 1612
columnas         id,dengue
mismo_orden           True
minimo            0.976574
media_h1_2025        65.75
media_h2_2025         65.7
dtype: object

## Lectura

La meta no baja el MSE historico de B ni en H1 (4181 contra 3534) ni en H2 (5348 contra 5164). En 2025 el resto del interior queda cerca del nivel implicado por las sondas, la Costa queda algo alta y Valle sigue cerca del nivel de B, no en la correccion fuerte que sugieren las sondas. No conviene reemplazar el envio de 1646,98 con este archivo. El detalle esta en `notas.md`.
